# Stage 2 — selective parser + normalization
Fresh raw input → intent contract → 3B code generation → verification / 7B repair → reference evaluation.

Inputs: **stage2_dual_source.zip** and **selective_facts_1_5b.zip**. This notebook is independent of the other method's outputs and all previous parser/Stage 2 result files. No training is performed. Use a GPU with at least 12 GiB memory; T4 is supported by the preflight. Only one model is resident at a time. With two GPUs the generator uses GPU 1; parser/repair use GPU 0.

Selective facts uses the corrected scope router plus operator normalization, without model correction. Tools uses the trained 1.5B tool adapter and its existing bounded loop, with normalization on contract replies. Every tool attempt/error, including placeholder queries, remains recorded. The generator, verifier, sandbox, repair limits and prompts are shared.

Run a two-task infrastructure smoke, then all 100 test tasks. A smoke can accept/reject/escalate: it checks infrastructure, not a quality threshold. The previously inspected test tasks provide diagnostic comparison, not independent research evidence. Download **stage2_dual_selective_proposed.zip** and the executed notebook. Compare both methods using the CPU collector from this same pack.

**Kaggle:** attach both required ZIPs as private datasets, enable Internet and T4 GPU (T4 x2 also works), then Save Version → Run All.


In [ ]:
import datetime,hashlib,json,os,shutil,subprocess,sys,zipfile
from pathlib import Path
INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working')
WORK.mkdir(parents=True,exist_ok=True)
RUN_ID=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR=WORK/('run_'+RUN_ID);RUN_DIR.mkdir()
DRIVE_RESULTS=None
def save_results():
    if DRIVE_RESULTS is not None:
        DRIVE_RESULTS.mkdir(parents=True,exist_ok=True)
        for p in RUN_DIR.iterdir():
            if p.is_file() and p.suffix in ['.zip','.log','.json','.txt']:shutil.copy2(p,DRIVE_RESULTS/p.name)
        print('Drive results:',DRIVE_RESULTS)
print('Run folder:',RUN_DIR)

METHOD='selective'
for name in ['stage2_dual_source.zip','selective_facts_1_5b.zip']:
    matches=list(INPUT.rglob(name));print(('found   ' if matches else 'MISSING ')+name,*matches)


In [ ]:
MANIFEST_SHA='d9e67c22c047a06aa7996efa2bde5f700c3c8dfdae46b65ad8220fbff609c335'
ROOT=WORK/('stage2_dual_source_'+MANIFEST_SHA[:12])
def unpack(raw,read):
    assert hashlib.sha256(raw).hexdigest()==MANIFEST_SHA,'Wrong source ZIP version'
    manifest=json.loads(raw);assert manifest['experiment']=='stage2-dual-parser-v1'
    for name,sha in manifest['sha256'].items():
        relative=Path(name);assert not relative.is_absolute() and '..' not in relative.parts
        data=read(name);assert hashlib.sha256(data).hexdigest()==sha,f'Changed source/input: {name}'
        path=ROOT/relative;path.parent.mkdir(parents=True,exist_ok=True)
        if path.exists():assert path.read_bytes()==data,f'Changed working source: {name}'
        else:path.write_bytes(data)
    sidecar=ROOT/'_stage2_dual_manifest.json'
    if sidecar.exists():assert sidecar.read_bytes()==raw,'Changed source manifest'
    else:sidecar.write_bytes(raw)
    print('Verified fresh-run source/data:',len(manifest['sha256']),'files',MANIFEST_SHA)
found=False
for path in INPUT.rglob('manifest.json'):
    if hashlib.sha256(path.read_bytes()).hexdigest()==MANIFEST_SHA:
        unpack(path.read_bytes(),lambda name:(path.parent/name).read_bytes());found=True;break
if not found:
    for path in INPUT.rglob('stage2_dual_source.zip'):
        with zipfile.ZipFile(path) as z:
            assert len(z.namelist())==len(set(z.namelist()))
            prefix='stage2_dual_source/';name=prefix+'manifest.json'
            if name not in z.namelist():continue
            raw=z.read(name)
            if hashlib.sha256(raw).hexdigest()!=MANIFEST_SHA:continue
            unpack(raw,lambda name:z.read(prefix+name));found=True;break
assert found,'Attach/place stage2_dual_source.zip in the input folder.'
PACKAGES=WORK/'stage2_dual_packages'
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(PACKAGES),'-r',str(ROOT/'requirements.txt')],check=True)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[ROOT,ROOT/'src',PACKAGES])),PYTHONUNBUFFERED='1',TOKENIZERS_PARALLELISM='false')


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.20.2','peft==0.21.2','bitsandbytes==0.50.2','accelerate==1.15.0','tokenizers==0.23.3','safetensors==0.8.0','huggingface-hub==1.33.0'],check=True)
subprocess.run([sys.executable,'-m','eval.stage2_independent_v2.environment','--prepare'],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-c',"import torch; assert torch.cuda.is_available(); assert all(torch.cuda.get_device_properties(i).total_memory/2**30>=12 for i in range(torch.cuda.device_count())); print(torch.__version__,[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-m','finetune.selective_facts.attention'],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-m','eval.stage2_dual.preflight'],cwd=ROOT,env=ENV,check=True)
path_file=RUN_DIR/'adapter_path.txt'
subprocess.run([sys.executable,'-m','eval.stage2_dual.adapters','--method',METHOD,'--input',str(INPUT),'--work',str(WORK),'--out',str(path_file)],cwd=ROOT,env=ENV,check=True)
ADAPTER=Path(path_file.read_text());save_results()


In [ ]:
def execute(smoke):
    name='stage2_dual_'+METHOD+'_proposed'+('_smoke' if smoke else '')
    out=RUN_DIR/name;assert not out.exists(),'Use a fresh run folder.'
    command=[sys.executable,'-m','eval.stage2_dual.runner','--method',METHOD,'--preset','proposed','--adapter',str(ADAPTER),'--out',str(out),'--packages',str(PACKAGES)]
    if smoke:command+=['--smoke']
    log=RUN_DIR/(name+'.log')
    try:
        with log.open('w') as f:
            process=subprocess.Popen(command,cwd=ROOT,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in process.stdout:
                print(line,end='');f.write(line);f.flush()
            code=process.wait()
    finally:save_results()
    assert code==0,f'Run failed: inspect {log} and the partial ZIP. A partial job is not a result.'
    assert json.loads((out/'completion.json').read_text())['complete']
    assert len(list((out/'tasks').glob('*.json')))==(2 if smoke else 100)
    return out.with_suffix('.zip')
SMOKE=execute(True)
print('Two-task infrastructure smoke completed. Next cell runs 100 test tasks freshly.')


In [ ]:
BUNDLE=execute(False)
from IPython.display import display,FileLink
display(FileLink(str(BUNDLE)))
if DRIVE_RESULTS is not None:print('Drive copy:',DRIVE_RESULTS/BUNDLE.name)
print('COMPLETE: download the full ZIP and this executed notebook. Run the other method independently, then the CPU collector.')
